# GLP-1 Data Cleaning & Integration
This notebook prepares and integrates multiple healthcare data sources
for exploratory data analysis (EDA) of GLP-1 Adverse Effect.

### Data Sources
1. FDA FAERS

### Workflow
Data acquisition -> Inspection -> Cleaning -> grouped 80/20 split -> Fitting all three models

In [2]:
%pip install pandas numpy matplotlib requests scikit-learn ipykernel

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import requests

# 1. FAERS Data

## 1.1 Data file Import

In [4]:
# Setting working directory, so I can work on desktop and later move file and it still works (makes path dynamic)
current_path = os.getcwd()
os.chdir(current_path)
print(f"Working Directory Set To: {current_path}")

Working Directory Set To: /Users/siennasmacbookpro/Desktop/DS5003_26FA/GROUP PROJECT


## 1.2 First Inspection

In [5]:
faers_df = pd.read_csv('glp1_faers_complete_2018_2025.csv')

In [6]:
display(faers_df.head())
print(faers_df.shape)
print(faers_df.columns.tolist())

,ingredient_query,safetyreportid,safetyreportversion,receivedate,receiptdate,occurcountry,primarysourcecountry,reporter_qualification,serious,hospitalized,...,patient_age_years,patient_sex,patient_weight_kg,matching_medicinal_products,matching_indications,matching_routes,matching_drug_roles,reaction_terms,n_reactions,n_drugs_in_report
0,Dulaglutide,14340470,1,20180101,20180101,US,US,Physician,0,0,...,NaN,Female,64.85,TRULICITY,TYPE 2 DIABETES MELLITUS,050,Concomitant,Decreased appetite | Dysgeusia | Insomnia | Na...,6,10
1,Dulaglutide,14341380,1,20180102,20180102,US,US,Consumer/non-health professional,0,0,...,66.0,Female,NaN,TRULICITY,TYPE 2 DIABETES MELLITUS,058,Suspect,Blood glucose decreased | Malaise,2,3
2,Dulaglutide,14342154,2,20180102,20180105,IL,IL,Physician,1,1,...,NaN,Male,NaN,TRULICITY,TYPE 2 DIABETES MELLITUS,065,Suspect,Diabetic ketoacidosis | Nausea,2,4
3,Dulaglutide,14342607,1,20180102,20180102,US,US,Consumer/non-health professional,0,0,...,NaN,Female,NaN,TRULICITY,PRODUCT USED FOR UNKNOWN INDICATION,065,Suspect,Abdominal distension | Abdominal pain upper,2,2
4,Dulaglutide,14342663,1,20180102,20180102,US,US,Consumer/non-health professional,0,0,...,64.0,Female,NaN,TRULICITY,TYPE 2 DIABETES MELLITUS,058,Suspect,Chills | Nausea,2,1


(281871, 27)
['ingredient_query', 'safetyreportid', 'safetyreportversion', 'receivedate', 'receiptdate', 'occurcountry', 'primarysourcecountry', 'reporter_qualification', 'serious', 'hospitalized', 'death', 'life_threatening', 'disabling', 'congenital_anomaly', 'other_serious', 'patient_age', 'patient_age_unit', 'patient_age_years', 'patient_sex', 'patient_weight_kg', 'matching_medicinal_products', 'matching_indications', 'matching_routes', 'matching_drug_roles', 'reaction_terms', 'n_reactions', 'n_drugs_in_report']


## 1.3 Initial Inspection

In [7]:
faers_df.head()

,ingredient_query,safetyreportid,safetyreportversion,receivedate,receiptdate,occurcountry,primarysourcecountry,reporter_qualification,serious,hospitalized,...,patient_age_years,patient_sex,patient_weight_kg,matching_medicinal_products,matching_indications,matching_routes,matching_drug_roles,reaction_terms,n_reactions,n_drugs_in_report
0,Dulaglutide,14340470,1,20180101,20180101,US,US,Physician,0,0,...,NaN,Female,64.85,TRULICITY,TYPE 2 DIABETES MELLITUS,050,Concomitant,Decreased appetite | Dysgeusia | Insomnia | Na...,6,10
1,Dulaglutide,14341380,1,20180102,20180102,US,US,Consumer/non-health professional,0,0,...,66.0,Female,NaN,TRULICITY,TYPE 2 DIABETES MELLITUS,058,Suspect,Blood glucose decreased | Malaise,2,3
2,Dulaglutide,14342154,2,20180102,20180105,IL,IL,Physician,1,1,...,NaN,Male,NaN,TRULICITY,TYPE 2 DIABETES MELLITUS,065,Suspect,Diabetic ketoacidosis | Nausea,2,4
3,Dulaglutide,14342607,1,20180102,20180102,US,US,Consumer/non-health professional,0,0,...,NaN,Female,NaN,TRULICITY,PRODUCT USED FOR UNKNOWN INDICATION,065,Suspect,Abdominal distension | Abdominal pain upper,2,2
4,Dulaglutide,14342663,1,20180102,20180102,US,US,Consumer/non-health professional,0,0,...,64.0,Female,NaN,TRULICITY,TYPE 2 DIABETES MELLITUS,058,Suspect,Chills | Nausea,2,1


In [8]:
faers_df.describe()


,safetyreportid,safetyreportversion,receivedate,receiptdate,serious,hospitalized,death,life_threatening,disabling,congenital_anomaly,other_serious,patient_age,patient_age_unit,patient_age_years,patient_weight_kg,n_reactions,n_drugs_in_report
count,2.818710e+05,281871.000000,2.818710e+05,2.818710e+05,281871.000000,281871.000000,281871.000000,281871.000000,281871.000000,281871.000000,281871.000000,167180.000000,167187.000000,167179.000000,34430.000000,281871.000000,281871.000000
mean,2.248544e+07,1.314867,2.023030e+07,2.023135e+07,0.299850,0.124440,0.015312,0.013751,0.011253,0.000365,0.208798,171.998289,801.016329,57.520673,96.669378,2.721678,5.039883
std,3.233958e+06,1.248340,2.129838e+04,2.134181e+04,0.458193,0.330083,0.122791,0.116456,0.105484,0.019112,0.406450,1622.834102,0.224463,14.072838,29.200329,3.121819,10.734797
min,1.434023e+07,1.000000,2.018010e+07,2.018010e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,800.000000,0.000000,0.450000,1.000000,1.000000
25%,2.050227e+07,1.000000,2.022022e+07,2.022033e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,48.000000,801.000000,48.000000,78.000000,1.000000,1.000000
50%,2.349999e+07,1.000000,2.024021e+07,2.024030e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,59.000000,801.000000,59.000000,92.400000,2.000000,2.000000
75%,2.520394e+07,1.000000,2.025042e+07,2.025051e+07,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,68.000000,801.000000,68.000000,109.090000,3.000000,5.000000
max,2.621932e+07,94.000000,2.025123e+07,2.026063e+07,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,34287.000000,805.000000,947.000000,471.170000,95.000000,1232.000000


In [9]:
faers_df.dtypes

ingredient_query                   str
safetyreportid                   int64
safetyreportversion              int64
receivedate                      int64
receiptdate                      int64
occurcountry                       str
primarysourcecountry               str
reporter_qualification             str
serious                          int64
hospitalized                     int64
death                            int64
life_threatening                 int64
disabling                        int64
congenital_anomaly               int64
other_serious                    int64
patient_age                    float64
patient_age_unit               float64
patient_age_years              float64
patient_sex                        str
patient_weight_kg              float64
matching_medicinal_products        str
matching_indications               str
matching_routes                    str
matching_drug_roles                str
reaction_terms                     str
n_reactions              

In [10]:
# Check dataset dimensions and missing values

print("Rows:", len(faers_df))
print("Columns:", faers_df.shape[1])

faers_df.isnull().sum()

Rows: 281871
Columns: 27


ingredient_query                    0
safetyreportid                      0
safetyreportversion                 0
receivedate                         0
receiptdate                         0
occurcountry                     8458
primarysourcecountry              489
reporter_qualification           1348
serious                             0
hospitalized                        0
death                               0
life_threatening                    0
disabling                           0
congenital_anomaly                  0
other_serious                       0
patient_age                    114691
patient_age_unit               114684
patient_age_years              114692
patient_sex                         0
patient_weight_kg              247441
matching_medicinal_products         0
matching_indications            20530
matching_routes                 41819
matching_drug_roles                 0
reaction_terms                      0
n_reactions                         0
n_drugs_in_r

In [11]:
# Check unique safety reports and duplicate rows

print("Rows:", len(faers_df))
print("Unique safety report IDs:", faers_df["safetyreportid"].nunique())
print("Exact duplicate rows:", faers_df.duplicated().sum())

Rows: 281871
Unique safety report IDs: 276902
Exact duplicate rows: 0


In [12]:
# Inspect safety reports that appear more than once

report_counts = faers_df["safetyreportid"].value_counts()

print("Report IDs appearing more than once:", (report_counts > 1).sum())
print("Maximum rows for one report ID:", report_counts.max())

report_counts[report_counts > 1].head(10)

Report IDs appearing more than once: 4388
Maximum rows for one report ID: 6


safetyreportid
24162132    6
23639121    5
24101468    5
24219551    5
22095022    4
22188513    4
22857577    4
24176911    4
24200207    4
24219421    4
Name: count, dtype: int64

#### Inspect ONE IDs-24162132

In [13]:
# Inspect multiple versions of one safety report

faers_df.loc[
    faers_df["safetyreportid"] == 24162132,
    [
        "safetyreportid",
        "safetyreportversion",
        "receivedate",
        "receiptdate",
        "ingredient_query",
        "reaction_terms"
    ]
].sort_values("safetyreportversion")

,safetyreportid,safetyreportversion,receivedate,receiptdate,ingredient_query,reaction_terms
65337,24162132,4,20240801,20260120,Dulaglutide,Abdominal pain | Constipation | Gastrointestin...
76318,24162132,4,20240801,20260120,Exenatide,Abdominal pain | Constipation | Gastrointestin...
93936,24162132,4,20240801,20260120,Liraglutide,Abdominal pain | Constipation | Gastrointestin...
140059,24162132,4,20240801,20260120,Tirzepatide,Abdominal pain | Constipation | Gastrointestin...
220298,24162132,4,20240801,20260120,Lixisenatide,Abdominal pain | Constipation | Gastrointestin...
257625,24162132,4,20240801,20260120,Semaglutide,Abdominal pain | Constipation | Gastrointestin...


In [14]:
# Inspect drug information for one repeated safety report

faers_df.loc[
    faers_df["safetyreportid"] == 24162132,
    [
        "safetyreportid",
        "ingredient_query",
        "matching_medicinal_products",
        "matching_drug_roles"
    ]
]

,safetyreportid,ingredient_query,matching_medicinal_products,matching_drug_roles
65337,24162132,Dulaglutide,DULAGLUTIDE | TRULICITY,Concomitant | Suspect
76318,24162132,Exenatide,BYETTA,Concomitant
93936,24162132,Liraglutide,SAXENDA | VICTOZA | XULTOPHY 100/3.6,Concomitant
140059,24162132,Tirzepatide,MOUNJARO | TIRZEPATIDE | ZEPBOUND,Concomitant
220298,24162132,Lixisenatide,SOLIQUA 100/33,Concomitant
257625,24162132,Semaglutide,OZEMPIC | RYBELSUS,Concomitant


## 1.4 Data Cleaning -FAERS

In [15]:
# Convert FAERS date variables to datetime

faers_df["receivedate"] = pd.to_datetime(
    faers_df["receivedate"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

faers_df["receiptdate"] = pd.to_datetime(
    faers_df["receiptdate"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

faers_df[["receivedate", "receiptdate"]].dtypes

receivedate    datetime64[us]
receiptdate    datetime64[us]
dtype: object

In [16]:
# Check missing dates after conversion

faers_df[["receivedate", "receiptdate"]].isnull().sum()

receivedate    0
receiptdate    0
dtype: int64

In [17]:
# Check GLP-1 drug names and frequencies

faers_df["ingredient_query"].value_counts()

ingredient_query
Tirzepatide     121302
Dulaglutide      73876
Semaglutide      60958
Liraglutide      20042
Lixisenatide      3098
Exenatide         2595
Name: count, dtype: int64

### Standardize the merge variable ('glp1_drug')

In [18]:
# Create standardized GLP-1 drug variable for later matching

faers_df["glp1_drug"] = (
    faers_df["ingredient_query"]
    .str.strip()
    .str.lower()
)

faers_df["glp1_drug"].value_counts()

glp1_drug
tirzepatide     121302
dulaglutide      73876
semaglutide      60958
liraglutide      20042
lixisenatide      3098
exenatide         2595
Name: count, dtype: int64

### Inspect Patient age 

In [19]:
# Inspect patient age in years

faers_df["patient_age_years"].describe()

count    167179.000000
mean         57.520673
std          14.072838
min           0.000000
25%          48.000000
50%          59.000000
75%          68.000000
max         947.000000
Name: patient_age_years, dtype: float64

In [20]:
# Inspect implausible patient ages

faers_df.loc[
    faers_df["patient_age_years"] > 120,
    [
        "safetyreportid",
        "glp1_drug",
        "patient_age",
        "patient_age_unit",
        "patient_age_years"
    ]
].sort_values("patient_age_years", ascending=False)

,safetyreportid,glp1_drug,patient_age,patient_age_unit,patient_age_years
251724,23725157,semaglutide,947.0,801.0,947.0
238387,21873994,semaglutide,40.0,800.0,400.0


### Inspect 'patient_age_unit'

In [21]:
# Inspect patient age units and their values

faers_df.groupby("patient_age_unit").agg(
    count=("patient_age", "count"),
    min_age=("patient_age", "min"),
    max_age=("patient_age", "max")
).sort_index()

,count,min_age,max_age
patient_age_unit,,,
800.0,287,3.0,40.0
801.0,165605,0.0,947.0
802.0,411,2.0,1074.0
803.0,26,4.0,4532.0
804.0,849,0.0,34287.0
805.0,1,1.0,1.0


### Identify 800-805 meaning in 'patient_age_unit'

In [22]:
# Compare original age, unit code, and converted age

faers_df[
    ["patient_age", "patient_age_unit", "patient_age_years"]
].groupby("patient_age_unit").head(5).sort_values("patient_age_unit")

,patient_age,patient_age_unit,patient_age_years
1928,7.0,800.0,70.0000
6376,8.0,800.0,80.0000
4729,8.0,800.0,80.0000
3581,5.0,800.0,50.0000
3348,9.0,800.0,90.0000
1,66.0,801.0,66.0000
8,54.0,801.0,54.0000
7,59.0,801.0,59.0000
6,51.0,801.0,51.0000
4,64.0,801.0,64.0000


### Flag impalusible ages

In [23]:
# Flag implausible patient ages

faers_df["age_implausible"] = faers_df["patient_age_years"] > 120

print("Implausible ages:", faers_df["age_implausible"].sum())

faers_df.loc[
    faers_df["age_implausible"],
    ["safetyreportid", "glp1_drug", "patient_age",
     "patient_age_unit", "patient_age_years"]
]

Implausible ages: 2


,safetyreportid,glp1_drug,patient_age,patient_age_unit,patient_age_years
238387,21873994,semaglutide,40.0,800.0,400.0
251724,23725157,semaglutide,947.0,801.0,947.0


In [24]:
# Create cleaned patient age variable

faers_df["patient_age_years_clean"] = faers_df["patient_age_years"]

# Set implausible ages to missing
faers_df.loc[
    faers_df["patient_age_years"] > 120,
    "patient_age_years_clean"
] = np.nan

# Check cleaned patient age
faers_df["patient_age_years_clean"].describe()

count    167177.000000
mean         57.513303
std          13.878505
min           0.000000
25%          48.000000
50%          59.000000
75%          68.000000
max         104.000000
Name: patient_age_years_clean, dtype: float64

#### Patient Age Validation

Patient age values were examined for plausibility. Two records had derived ages greater than 120 years (400 and 947 years). 
Two age values exceeding 120 years were set to missing; the reports and original age fields were retained in a new `patient_age_years_clean` variable. 


### Flag negative ages and ages above 120

In [25]:
age = pd.to_numeric(faers_df["patient_age_years"], errors="coerce")

faers_df["age_implausible"] = age.lt(0) | age.gt(120)

faers_df["patient_age_years_clean"] = age.mask(
    faers_df["age_implausible"]
)

# Preserve the distinction between unreported and invalid age
faers_df["age_missing_original"] = age.isna().astype(int)
faers_df["age_unavailable_clean"] = (
    faers_df["patient_age_years_clean"].isna().astype(int)
)

In [28]:
clean_age = faers_df["patient_age_years_clean"]

print("Negative ages before cleaning:", (age < 0).sum())
print("Ages above 120 before cleaning:", (age > 120).sum())
print("Invalid ages remaining:",
      ((clean_age < 0) | (clean_age > 120)).sum())

print("\nCleaned age summary:")
print(clean_age.describe())

print("\nAge unavailable after cleaning:")
print(faers_df["age_unavailable_clean"].value_counts(dropna=False))

Negative ages before cleaning: 0
Ages above 120 before cleaning: 2
Invalid ages remaining: 0

Cleaned age summary:
count    167177.000000
mean         57.513303
std          13.878505
min           0.000000
25%          48.000000
50%          59.000000
75%          68.000000
max         104.000000
Name: patient_age_years_clean, dtype: float64

Age unavailable after cleaning:
age_unavailable_clean
0    167177
1    114694
Name: count, dtype: int64


0= a usable age is availble.; 1= age is missing or was flagged as invalid.

## 1.5 80/20 split

In [29]:
from sklearn.linear_model import LogisticRegression

In [32]:
from sklearn.model_selection import GroupShuffleSplit

model_df = faers_df.copy()
model_df["report_year"] = model_df["receivedate"].dt.year

categorical_cols = [
    "patient_sex",
    "glp1_drug",
    "matching_drug_roles",
    "reporter_qualification",
    "report_year"
]
# Treat missing categories as an explicit category
for col in categorical_cols:
    model_df[col] = model_df[col].fillna("Unknown").astype(str)

assert model_df["safetyreportid"].notna().all()
assert model_df["hospitalized"].isin([0, 1]).all()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    splitter.split(
        model_df,
        groups=model_df["safetyreportid"]
    )
)

train = model_df.iloc[train_index].copy()
test = model_df.iloc[test_index].copy()

# Confirm no report appears in both sets
assert set(train["safetyreportid"]).isdisjoint(
    set(test["safetyreportid"])
)

for name, data in [("Training", train), ("Test", test)]:
    print(
        f"{name}: {len(data):,} rows; "
        f"hospitalization = {data['hospitalized'].mean():.2%}"
    )

Training: 225,518 rows; hospitalization = 12.46%
Test: 56,353 rows; hospitalization = 12.37%


## 1.6 Fit three models

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer                            # files missing nunmeric values.
from sklearn.preprocessing import OneHotEncoder, StandardScaler     # OneHotEncoder for categorical variables, StandardScaler for numeric variables.
from sklearn.linear_model import LogisticRegression                 # imports the LR.
from sklearn.metrics import (
    roc_auc_score,                                                  # how well predictions rank hospitalized reports above non-hospitalized reports.
    average_precision_score,                                        # precision-recall AUC
    log_loss,                                                       # evaluates predicted prob. (Lower is better)
    brier_score_loss                                                # MSE (lower is better)
)

model_specs = {
    "1: Baseline": [],

    "2: Baseline + age-unavailable indicator": [
        "age_unavailable_clean"
    ],

    "3: Baseline + age-unavailable indicator + age": [
        "age_unavailable_clean",
        "patient_age_years_clean"
    ]
}

models = {}
results = []
test_predictions = {}

y_train = train["hospitalized"]
y_test = test["hospitalized"]

for name, numeric_cols in model_specs.items():

    transformers = [
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]

    if numeric_cols:
        numeric_pipeline = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ])

        transformers.append(
            ("numeric", numeric_pipeline, numeric_cols)
        )

    model = Pipeline([
        ("preprocess", ColumnTransformer(transformers)),
        ("logistic", LogisticRegression(max_iter=2000))
    ])

    features = categorical_cols + numeric_cols

    # Imputation and scaling are learned from training data only
    model.fit(train[features], y_train)

    probabilities = model.predict_proba(test[features])[:, 1]

    models[name] = model
    test_predictions[name] = probabilities

    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average precision": average_precision_score(
            y_test, probabilities
        ),
        "Log loss": log_loss(y_test, probabilities),
        "MSE (Brier score)": brier_score_loss(y_test, probabilities)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(5))

,ROC-AUC,Average precision,Log loss,MSE (Brier score)
Model,,,,
1: Baseline,0.74815,0.30606,0.33025,0.09752
2: Baseline + age-unavailable indicator,0.75404,0.32891,0.32581,0.09571
3: Baseline + age-unavailable indicator + age,0.75415,0.32970,0.32573,0.09568
